# Clasificación - Predicción de Dirección del Oro

**Autor**: Juan (Scrum Master)

**Objetivo**: Predecir si el precio del oro (GC=F) subirá o bajará al día siguiente.

**Tipo de problema**: Clasificación binaria (Up = 1, Down = 0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
%matplotlib inline

## 1. Carga de datos

In [ ]:
ticker = 'GC=F'
start_date = '2015-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

df = yf.download(ticker, start=start_date, end=end_date)
df.head()

In [ ]:
print(f'Registros: {len(df)}')
print(f'Columnas: {df.columns.tolist()}')
print(f'Rango de fechas: {df.index[0].date()} a {df.index[-1].date()}')
print(f'Nulos:\n{df.isnull().sum()}')

In [ ]:
df.describe()

## 2. Feature engineering

Creamos la variable target y features relevantes para clasificación.

In [ ]:
df = df.copy()

df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df['Returns'] = df['Close'].pct_change()
df['Range'] = df['High'] - df['Low']
df['MA_5'] = df['Close'].rolling(5).mean()
df['MA_10'] = df['Close'].rolling(10).mean()
df['Volume_MA_5'] = df['Volume'].rolling(5).mean()
df['Volatility'] = df['Returns'].rolling(10).std()
df['Close_MA5_ratio'] = df['Close'] / df['MA_5']
df.dropna(inplace=True)

print(df[['Close', 'Target', 'Returns', 'Range', 'Close_MA5_ratio']].head())

In [ ]:
print(f'Distribución del target:')
print(df['Target'].value_counts())
print(f'\nProporción:\n{df["Target"].value_counts(normalize=True)}')

## 3. Preprocesamiento

Dividimos en train/test y escalamos. El scaler se ajusta SOLO en train para evitar data leakage.

In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Returns', 'Range',
            'MA_5', 'MA_10', 'Volume_MA_5', 'Volatility', 'Close_MA5_ratio']

X = df[features]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape[0]} registros')
print(f'Test: {X_test_scaled.shape[0]} registros')

## 4. Baseline (DummyClassifier)

Modelo de referencia que siempre predice la clase mayoritaria. Sirve para saber si nuestros modelos aportan valor real.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

print('Baseline - Resultados:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dummy):.4f}')
print(f'F1-score: {f1_score(y_test, y_pred_dummy):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, y_pred_dummy, target_names=["Down", "Up"])}')

## 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print('Random Forest - Resultados en TEST:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-score: {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, y_pred_rf, target_names=["Down", "Up"])}')

## 6. Comparativa Baseline vs Random Forest

In [ ]:
results = pd.DataFrame({
    'Modelo': ['Baseline (siempre Up)', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_dummy), accuracy_score(y_test, y_pred_rf)],
    'F1-score': [f1_score(y_test, y_pred_dummy), f1_score(y_test, y_pred_rf)]
})
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(confusion_matrix(y_test, y_pred_dummy), annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Baseline - Matriz de Confusión')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('Random Forest - Matriz de Confusión')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

## 7. Detección de Overfitting

Comparamos rendimiento en train vs test para detectar sobreajuste.

In [ ]:
y_train_pred = rf.predict(X_train_scaled)

print('Random Forest - Comparación Train vs Test:')
print(f'Train Accuracy: {accuracy_score(y_train, y_train_pred):.4f}')
print(f'Test Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Train F1: {f1_score(y_train, y_train_pred):.4f}')
print(f'Test F1:  {f1_score(y_test, y_pred_rf):.4f}')

diff = accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_pred_rf)
if diff > 0.1:
    print(f'\n⚠️ Posible overfitting (diferencia: {diff:.4f})')
else:
    print(f'\n✅ Sin overfitting significativo (diferencia: {diff:.4f})')

In [ ]:
cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='f1')
print(f'Cross-validation F1: {cv_scores}')
print(f'Media CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 8. Importancia de Features

In [ ]:
importances = pd.DataFrame({
    'Feature': features,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importances, x='Importancia', y='Feature')
plt.title('Importancia de Features - Random Forest')
plt.tight_layout()
plt.show()

## 9. Conclusiones

- El modelo Random Forest supera al baseline, lo que indica que los features tienen poder predictivo.
- La métrica principal es **F1-score** porque las clases pueden estar desbalanceadas.
- Revisar si hay overfitting comparando train vs test.
- **Mejoras posibles**: añadir más features externas (USD index, tasas de interés), ajustar hiperparámetros, probar XGBoost.